# Medical Accesibility Scoring and Kansas County Maps

This notebook explores healthcare accessibility across Kansas. It combines patient and department information, calculates county-level summaries of healthcare locations and patient counts, and examines transportation needs, financial strain, and social connections. It then develops exploratory Medical Access Scores and visualizes the results using interactive Folium county and Census-tract maps.

Inputs: `foot.csv`, `departments.csv`, `sd_clean.csv`, `kansas_counties.geojson`, and `kansas_tracts.geojson`. (restricted ASA Datafest data, not included)

Outputs: Html maps saved to outputs/.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
import webbrowser

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / 'data').is_dir() and (p / 'notebooks').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root not found.')

RAW_DIR = PROJECT_ROOT / 'data' / 'private' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'private' / 'processed'
GEO_DIR = PROJECT_ROOT / 'data' / 'geospatial'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACTS_PATH = GEO_DIR / 'kansas_tracts.geojson'
COUNTIES_PATH = GEO_DIR / 'kansas_counties.geojson'

In [ ]:
clean_df=pd.read_csv(PROCESSED_DIR / 'foot.csv')
clean_df.head()

In [ ]:
map_df=clean_df.copy()
map_df['access_score']=0
map_df['GEOID']=map_df["GEOID"].astype(str)
map_df["county"] = map_df["GEOID"].str[:5]

map_df.head()
map_df.shape

In [ ]:
gdf = gpd.read_file("/Users/deancallahan/Downloads/tl_2025_20_tract/tl_2025_20_tract.shp")

gdf.to_file("kansas_tracts.geojson", driver="GeoJSON")

# Load US counties
county_df = gpd.read_file('/Users/deancallahan/Downloads/tl_2025_us_county/tl_2025_us_county.shp')
gdf_kansas = county_df[county_df["STATEFP"] == "20"]
gdf_kansas.to_file("kansas_counties.geojson", driver="GeoJSON")

#Load Kansas Blocks
#kansas_blocks = gpd.read_file("/Users/deancallahan/Downloads/tl_2025_20_tabblock20/tl_2025_20_tabblock20.shp")

#kansas_blocks.to_file("kansas_blocks.geojson", driver="GeoJSON")


In [ ]:
# Load Kansas Census tracts
gdf = gpd.read_file(GEO_DIR / "kansas_tracts.geojson")

# Load Kansas counties
gdf_kansas = gpd.read_file(GEO_DIR / "kansas_counties.geojson")

In [ ]:
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=map_df,
    columns=["GEOID", "access_score"],
    key_on="feature.properties.GEOID",
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=str(COUNTIES_PATH),
    name="Counties",  
    data=map_df,
    columns=["GEOID", "access_score"],
    key_on="feature.properties.GEOID",
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME"],         
        aliases=["County:"],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "01_initial_map.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
dept_df=pd.read_csv(RAW_DIR / 'departments.csv')
dept_df.head()

In [ ]:
map_df.head()

In [ ]:
dept_df=dept_df.dropna(subset=['CensusTract'])
dept_df.head()

In [ ]:
dept_df['CensusTract']=dept_df['CensusTract'].astype(float).astype('Int64').astype(str)
dept_df.head()

In [ ]:
map_df.head()
map_df['GEOID'] = map_df['GEOID'].astype(str)

# create tract (first 11 digits)
map_df['tract'] = map_df['GEOID'].str[:11]

# create block (last digit)
map_df['block'] = map_df['GEOID'].str[-1]
map_df.head()

In [ ]:
merged_df=pd.merge(map_df,dept_df,left_on='tract',right_on='CensusTract')
merged_df['county'] = merged_df['GEOID'].str[:5]

In [ ]:
df_unique = merged_df[['county', 'CENTLAT', 'CENTLON']].drop_duplicates()

result = df_unique.groupby('county').size().reset_index(name='num_locations')
result['county']=result['county'].astype(str)
county_df_w_loc=result.copy()
#county_df_w_loc.rename(columns={"county": "GEOID"}, inplace=True)
county_df_w_loc


In [ ]:
merged_df = merged_df.merge(result,
    on='county',
    how='left'
)
merged_df.head()

In [ ]:
df_unique = merged_df[['tract', 'CENTLAT', 'CENTLON']].drop_duplicates()
result = df_unique.groupby('tract').size().reset_index(name='num_locations')

In [ ]:
temp=merged_df.copy()
temp['county']=temp['GEOID'].str[:5]
county_locations = (
    temp[['county', 'CENTLAT', 'CENTLON']]
    .drop_duplicates()
    .groupby('county')
    .size()
    .reset_index(name='num_locations')
)

county_df = county_locations.rename(columns={'county': 'GEOID'})
#county_locations = (
#    merged_df[['county', 'CENTLAT', 'CENTLON']]
 #   .drop_duplicates()
  ## .size()
    ##.reset_index(name='num_locations')
#)
county_df

In [ ]:


# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=merged_df,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=str(COUNTIES_PATH),
    name="Counties",  
    data=county_df,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME"],         
        aliases=["County:"],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "02_location_counts.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
county_df.head()

In [ ]:
merged_df.columns

In [ ]:
df_unique = merged_df[['County', 'PatientDurableKey']].drop_duplicates()

pop_result = df_unique.groupby('County').size().reset_index(name='num_patients')
pop_result['County']=pop_result['County'].astype(str)
pop_result

In [ ]:
merged_df = merged_df.merge(pop_result,
    on='County',
    how='left'
)
merged_df.head()

In [ ]:
temp=merged_df[['CensusTract','num_locations','num_patients']]
temp['GEOID'] = temp['CensusTract'].astype(str).str[:5]
temp


In [ ]:
county_df_two = (
    temp.groupby('GEOID')
    .agg(
        num_locations=('num_locations', 'mean'),        
        num_patients=('num_patients', 'max')     
    )
    .reset_index()
)
county_df_two

In [ ]:
county_df_two['med_access_score']=0
county_df_two['med_access_score']=(county_df_two['num_locations']/county_df_two['num_patients'])*1000
county_df_two

In [ ]:
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=merged_df,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=str(COUNTIES_PATH),
    name="Counties",  
    data=county_df_two,
    columns=["GEOID", "med_access_score"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME"],         
        aliases=["County:"],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "03_location_patient_ratio.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
import geopandas as gpd

# load counties
gdf_counties = gpd.read_file(COUNTIES_PATH)

# make sure GEOID types match
gdf_counties['GEOID'] = gdf_counties['GEOID'].astype(str)
county_df_two['GEOID'] = county_df_two['GEOID'].astype(str)

# merge score into geojson
gdf_counties = gdf_counties.merge(
    county_df_two[['GEOID', 'med_access_score']],
    on='GEOID',
    how='left'
)

In [ ]:
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=merged_df,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=gdf_counties,
    name="Counties",  
    data=county_df_two,
    columns=["GEOID", "med_access_score"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME","med_access_score"],         
        aliases=["County:",'Access Score'],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "04_county_access_score.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
merged_df_transport=merged_df.copy()
merged_df_transport.head()

In [ ]:
transportation = pd.read_csv(PROCESSED_DIR / 'sd_clean.csv')
transportation.info()

In [ ]:
transportation.drop(transportation.columns[[0]], axis=1, inplace=True)
transportation.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Yes'
transportation_Filter = transportation[transportation['AnswerText'] == 'Yes']

transportation_Filter.info()

In [ ]:
T_merged = pd.merge(merged_df_transport, transportation_Filter, on=['PatientDurableKey'], how='inner')
T_merged.info()

In [ ]:
print(T_merged['Domain'])

In [ ]:
# 1. Filter for the specific domain and the 'yes' response
T_merged = T_merged[(T_merged['Domain'] == 'Transportation Needs') & (T_merged['AnswerText'] == 'Yes')]

# 2. Group by county and count unique patient IDs
county_counts_T = T_merged.groupby('County')['PatientDurableKey'].nunique().reset_index()

# 3. Rename columns for clarity
county_counts_T.columns = ['County', 'Unique_Patient_Count_Transport']

print(county_counts_T)

In [ ]:
import matplotlib.pyplot as plt
county_counts_T.plot(kind='bar', x='County', y='Unique_Patient_Count_Transport', figsize=(10, 6), color='skyblue', edgecolor='navy')
plt.title('Unique Patients with Transportation Needs by County')
plt.xlabel('County')
plt.ylabel('Number of Unique Patients')
plt.xticks(rotation=45) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
county_counts_T

In [ ]:
merged_df_three = merged_df_transport.merge(county_counts_T,
    on='County',
    how='left'
)
merged_df_three.head()

In [ ]:
temp=merged_df_three[['CensusTract','num_locations','num_patients','Unique_Patient_Count_Transport']]
temp['GEOID'] = temp['CensusTract'].astype(str).str[:5]
temp

In [ ]:
county_df_three = (
    temp.groupby('GEOID')
    .agg(
        num_locations=('num_locations', 'mean'),        
        num_patients=('num_patients', 'max'),     
        num_need_transport=('Unique_Patient_Count_Transport','max')
    )
    .reset_index()
)
county_df_three

In [ ]:
county_df_three['med_access_score']=0
county_df_three['med_access_score']=((county_df_three['num_locations']/county_df_three['num_patients']) -  (county_df_three['num_need_transport']/county_df_three['num_patients']))

county_df_three

In [ ]:
import geopandas as gpd

# load counties
gdf_counties = gpd.read_file(COUNTIES_PATH)

# make sure GEOID types match
gdf_counties['GEOID'] = gdf_counties['GEOID'].astype(str)
county_df_three['GEOID'] = county_df_three['GEOID'].astype(str)

# merge score into geojson
gdf_counties = gdf_counties.merge(
    county_df_three[['GEOID', 'med_access_score']],
    on='GEOID',
    how='left'
)

In [ ]:
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=merged_df,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=gdf_counties,
    name="Counties",  
    data=county_df_two,
    columns=["GEOID", "med_access_score"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME","med_access_score"],         
        aliases=["County:",'Access Score'],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "05_transportation_access.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
merged_df_financial=merged_df_three.copy()
merged_df_financial.head()

In [ ]:
financial = pd.read_csv(PROCESSED_DIR / 'sd_clean.csv')
financial.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Financial Needs'
Financial_Filter = financial[financial['Domain'] == 'Financial Resource Strain']
Financial_Filter.info()

In [ ]:
Financial_Filter.drop(Financial_Filter.columns[[0]], axis=1, inplace=True)
Financial_Filter.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Yes'
Financial_Filter = financial[financial['AnswerText'] == 'Yes']

Financial_Filter.info()

In [ ]:
Financial_Filter.drop(Financial_Filter.columns[[0]], axis=1, inplace=True)
Financial_Filter.info()

In [ ]:
Fin_merged = pd.merge(merged_df, Financial_Filter, on=['PatientDurableKey'], how='inner')
Fin_merged.info()

In [ ]:
# 1. Filter for the specific domain and the 'yes' response
Fin_merged = Fin_merged[(Fin_merged['Domain'] == 'Financial Resource Strain') & (Fin_merged['AnswerText'] == 'Yes')]

# 2. Group by county and count unique patient IDs
county_counts_F = Fin_merged.groupby('County')['PatientDurableKey'].nunique().reset_index()

# 3. Rename columns for clarity
county_counts_F.columns = ['County', 'Unique_Patient_Count_Financial']

print(county_counts_F)

In [ ]:
merged_df_four = merged_df_financial.merge(county_counts_F,
    on='County',
    how='left'
)
merged_df_four.head()

In [ ]:
temp=merged_df_four[['CensusTract','num_locations','num_patients','Unique_Patient_Count_Transport','Unique_Patient_Count_Financial']]
temp['GEOID'] = temp['CensusTract'].astype(str).str[:5]
temp

In [ ]:
county_df_four = (
    temp.groupby('GEOID')
    .agg(
        num_locations=('num_locations', 'mean'),        
        num_patients=('num_patients', 'max'),     
        num_need_transport=('Unique_Patient_Count_Transport','max'),
        num_financial=('Unique_Patient_Count_Financial','max')

    )
    .reset_index()
)
county_df_four

In [ ]:
county_df_four['med_access_score']=0
county_df_four['med_access_score']=((county_df_four['num_locations']/county_df_four['num_patients']) -  (county_df_four['num_need_transport']/county_df_four['num_patients'])) -  (county_df_four['num_need_transport']/county_df_four['num_patients'])

county_df_four

In [ ]:
import geopandas as gpd

# load counties
gdf_counties = gpd.read_file(COUNTIES_PATH)

# make sure GEOID types match
gdf_counties['GEOID'] = gdf_counties['GEOID'].astype(str)
county_df_four['GEOID'] = county_df_four['GEOID'].astype(str)

# merge score into geojson
gdf_counties = gdf_counties.merge(
    county_df_four[['GEOID', 'med_access_score']],
    on='GEOID',
    how='left'
)

In [ ]:
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=merged_df_four,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=gdf_counties,
    name="Counties",  
    data=county_df_four,
    columns=["GEOID", "med_access_score"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME","med_access_score"],         
        aliases=["County:",'Access Score'],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "06_financial_access.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
merged_df_social=merged_df_four.copy()
merged_df_social.head()

In [ ]:
social = pd.read_csv(PROCESSED_DIR / 'sd_clean.csv')
social.info()


In [ ]:
social.drop(social.columns[[0]], axis=1, inplace=True)
social.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Yes'
social_Filter = social[social['AnswerText'] == 'No']

social_Filter.info()

In [ ]:
social_Filter = social_Filter[(social_Filter['Domain'] == 'social connections') & (social_Filter['AnswerText'] == 'No')]

In [ ]:
S_merged = pd.merge(merged_df, social_Filter, on=['PatientDurableKey'], how='inner')
S_merged.info()

In [ ]:
print(S_merged['Domain'])

In [ ]:
# 1. Filter for the specific domain and the 'yes' response
S_merged = S_merged[(S_merged['Domain'] == 'social connections') & (S_merged['AnswerText'] == 'No')]

# 2. Group by county and count unique patient IDs
county_counts_S = S_merged.groupby('County')['PatientDurableKey'].nunique().reset_index()

# 3. Rename columns for clarity
county_counts_S.columns = ['County', 'Unique_Patient_Count_Social']

print(county_counts_S)

In [ ]:

merged_df_five = merged_df_social.merge(county_counts_S,
    on='County',
    how='left'
)
merged_df_five.head()

In [ ]:
temp=merged_df_five[['CensusTract','num_locations','num_patients','Unique_Patient_Count_Transport','Unique_Patient_Count_Financial','Unique_Patient_Count_Social']]
temp['GEOID'] = temp['CensusTract'].astype(str).str[:5]
temp

In [ ]:
county_df_five = (
    temp.groupby('GEOID')
    .agg(
        num_locations=('num_locations', 'mean'),        
        num_patients=('num_patients', 'max'),     
        num_need_transport=('Unique_Patient_Count_Transport','max'),
        num_financial=('Unique_Patient_Count_Financial','max'),
        num_social=('Unique_Patient_Count_Social','max')


    )
    .reset_index()
)
county_df_five

In [ ]:
county_df_five['med_access_score']=0
county_df_five['med_access_score']=(50*(county_df_five['num_locations']/county_df_five['num_patients']) +  20*(county_df_five['num_need_transport']/county_df_five['num_patients'])+ 20*(county_df_five['num_financial']/county_df_five['num_patients'])   + 10*(county_df_five['num_social']/county_df_five['num_patients']) )-1.0

county_df_five

In [ ]:
import geopandas as gpd

# load counties
gdf_counties = gpd.read_file(COUNTIES_PATH)

# make sure GEOID types match
gdf_counties['GEOID'] = gdf_counties['GEOID'].astype(str)
county_df_five['GEOID'] = county_df_five['GEOID'].astype(str)

# merge score into geojson
gdf_counties = gdf_counties.merge(
    county_df_five[['GEOID', 'med_access_score']],
    on='GEOID',
    how='left'
)

In [ ]:
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(TRACTS_PATH),
    name="Census Tracts",  
    data=merged_df_five,
    columns=["GEOID", "num_locations"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=gdf_counties,
    name="Counties",  
    data=county_df_five,
    columns=["GEOID", "med_access_score"],
    key_on="feature.properties.GEOID",
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME","med_access_score"],         
        aliases=["County:",'Access Score'],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
map_path = OUTPUT_DIR / "07_combined_access_score.html"
m2.save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())